In [14]:
import os
from database.db import Database
from database.ql import QLExecutor

In [15]:
os.system("rm -rf example.mydb")

0

In [16]:
db = Database("example.mydb")

In [17]:
db.open()

In [18]:
executor = QLExecutor(db)

In [19]:
text = """
CREATE TABLE users (
    id INTEGER,
    name TEXT,
    age INTEGER,
    height INTEGER,
    INDEX name,
    INDEX age,
    INDEX height,
    PRIMARY KEY id
);

INSERT INTO users (id, name, age, height) VALUES (1, 'Alice', 25, 165 + -(50 % 32));
INSERT INTO users (id, name, age, height) VALUES (2, 'Bob', 30, 175 - 50 % 32);
INSERT INTO users (id, name, age, height) VALUES (3, 'Cindy', 35, 185 - 50 % 32);
INSERT INTO users (id, name, age, height) VALUES (4, 'David', 40, 195 - 50 % 32);

SELECT id AS user_id, name, age, height FROM users
INDEX BY name >= 'Bob' AND name <= 'Dob'
FILTER BY age <= 40 LIMIT 10;
"""

In [20]:
recs = executor.execute(text)
print(recs)

# 3 records found
‖ user_id    ‖ name       ‖ age        ‖ height    ‖
| 2          | Bob        | 30         | 157       |
----------------------------------------------------
| 3          | Cindy      | 35         | 167       |
----------------------------------------------------
| 4          | David      | 40         | 177       |
----------------------------------------------------



In [21]:
recs = executor.execute("""
SELECT id AS user_id, age, name, height FROM users OFFSET 1 LIMIT 2;
""")
print(recs)

# 2 records found
‖ user_id    ‖ age        ‖ name       ‖ height    ‖
| 2          | 30         | Bob        | 157       |
----------------------------------------------------
| 3          | 35         | Cindy      | 167       |
----------------------------------------------------



In [22]:
recs = executor.execute("""
UPDATE users SET age = 50 INDEX BY name = 'Bob';

SELECT id AS user_id, name, age, height FROM users
FILTER BY age <= 40 LIMIT 10;
""")
print(recs)

# 3 records found
‖ user_id    ‖ name       ‖ age        ‖ height    ‖
| 1          | Alice      | 25         | 147       |
----------------------------------------------------
| 3          | Cindy      | 35         | 167       |
----------------------------------------------------
| 4          | David      | 40         | 177       |
----------------------------------------------------



In [23]:
recs = executor.execute("""
DELETE FROM users INDEX BY name = 'David';

SELECT id AS user_id, age, name FROM users
INDEX BY id > 1
FILTER BY age <= 40 LIMIT 10;
""")
print(recs)

# 1 record found
‖ user_id    ‖ age        ‖ name      ‖
| 3          | 35         | Cindy     |
---------------------------------------



In [24]:
recs = executor.execute("""
DROP TABLE users;
""")

In [25]:
try:
  recs = executor.execute("""
  SELECT id AS user_id, age, name FROM users
  INDEX BY id > 1
  FILTER BY age <= 40 LIMIT 10;
  """)
  print(recs)
except ValueError as e:
  print(e)

Table users does not exist


In [26]:
db.close()